# 扩展主题：模型量化（Quantization）

> **性质**：🔬 真做  ｜  **依赖**：ch05 模型权重

## 一句话

把模型的 fp32 权重压成 **int8/int4**，显存省 75%~87.5%，推理更快，质量损失可控——这是部署大模型到消费级硬件的关键技术。

## 为什么需要量化

| 精度 | 每参数字节 | 7B 模型显存 |
|------|----------|-----------|
| fp32 | 4 | 28 GB |
| fp16 | 2 | 14 GB |
| **int8** | **1** | **7 GB** |
| **int4** | **0.5** | **3.5 GB** ← 单卡可跑 |

> 量化的核心假设：权重的实际分布集中在小范围，用低位整数近似后误差很小。

## 量化原理（对称量化）

fp32 → int8：找最大绝对值 `max`，缩放系数 `scale = max / 127`，量化 `q = round(w / scale)`，反量化 `w ≈ q × scale`。

In [ ]:
import torch
import torch.nn as nn


def quantize_int8(weight):
    """对称量化：fp32 → int8。返回 (量化权重, 缩放系数)。"""
    max_val = weight.abs().max()
    scale = max_val / 127.0
    q_weight = torch.round(weight / scale).to(torch.int8)
    return q_weight, scale

def dequantize(q_weight, scale):
    """反量化：int8 → fp32。"""
    return q_weight.float() * scale

def quantize_int4(weight):
    """对称量化：fp32 → int4（范围 [-8, 7]）。"""
    max_val = weight.abs().max()
    scale = max_val / 7.0
    q_weight = torch.round(weight / scale).clamp(-8, 7).to(torch.int8)  # PyTorch 无 int4，用 int8 存
    return q_weight, scale


# 验证量化误差
torch.manual_seed(0)
W = torch.randn(128, 128) * 0.1

q8, s8 = quantize_int8(W)
q4, s4 = quantize_int4(W)
err8 = (W - dequantize(q8, s8)).abs().mean()
err4 = (W - dequantize(q4, s4)).abs().mean()

print(f"{'精度':<8} {'每参数字节':<10} {'量化误差(MAE)':<15}")
print("-" * 35)
print(f"{'fp32':<8} {4:<10} {'(基准)':<15}")
print(f"{'int8':<8} {1:<10} {err8.item():<15.6f}")
print(f"{'int4':<8} {0.5:<10} {err4.item():<15.6f}")
print("\n💡 int8 误差极小几乎无损；int4 误差增大但仍可用。")

## 2. 量化整个 GPT 并对比显存

In [ ]:
from src.gpt import GPTModel, GPT_CONFIG_124M

def quantize_model_int8(model):
    """把模型所有 Linear 的权重量化到 int8（反量化存回，模拟 int8 推理）。"""
    for module in model.modules():
        if isinstance(module, nn.Linear):
            q_w, scale = quantize_int8(module.weight.data)
            module.weight.data = dequantize(q_w, scale)

cfg = dict(GPT_CONFIG_124M)
cfg.update({"emb_dim": 128, "n_layers": 2, "n_heads": 4, "context_length": 32})
torch.manual_seed(123)
model_fp32 = GPTModel(cfg)
model_int8 = GPTModel(cfg)
quantize_model_int8(model_int8)

# 对比显存占用（参数内存）
def param_memory(model):
    return sum(p.numel() * p.element_size() for p in model.parameters())

mem_fp32 = param_memory(model_fp32)
# int8 模型的参数在反量化后仍是 fp32 存储，这里按「量化后存储」理论值算
print(f"fp32 模型参数内存: {mem_fp32/1024:.1f} KB")
print(f"int8 理论内存:     {mem_fp32/4/1024:.1f} KB（省 75%）")
print(f"int4 理论内存:     {mem_fp32/8/1024:.1f} KB（省 87.5%）")

# 对比输出差异
import tiktoken
tok = tiktoken.get_encoding("gpt2")
idx = torch.tensor([tok.encode("Hello")])
model_fp32.eval(); model_int8.eval()
with torch.no_grad():
    out_fp32 = model_fp32(idx)
    out_int8 = model_int8(idx)
diff = (out_fp32 - out_int8).abs().mean().item()
print(f"\n量化前后输出差异(MAE): {diff:.6f}（应很小）")

## 3. QLoRA：量化 + LoRA（单卡微调大模型）

把 4bit 量化的基座模型 + LoRA 微调 = **QLoRA**。这是工业界在消费级 GPU 上微调 7B+ 模型的标准方案。

> 量化省基座显存，LoRA 只训极小参数——两者结合让单张显卡也能微调大模型。详见附录 E 的 LoRA。

```
QLoRA = 4bit量化基座(冻结) + LoRA旁路(训练)
       ↓ 省显存              ↓ 少参数
       单卡跑得动 7B          微调高效
```

In [ ]:
# QLoRA 概念演示：量化基座 + 接 LoRA（复用附录 E 的 LoRA）
# 这里展示量化后的模型仍能正常接 LoRA 微调

class LoRALayer(nn.Module):
    def __init__(self, layer, r=8, alpha=16):
        super().__init__()
        self.layer = layer
        for p in layer.parameters(): p.requires_grad = False
        in_f, out_f = layer.in_features, layer.out_features
        self.scaling = alpha / r
        self.A = nn.Parameter(torch.randn(in_f, r) * 0.01)
        self.B = nn.Parameter(torch.zeros(r, out_f))
    def forward(self, x):
        return self.layer(x) + self.scaling * (x @ (self.A @ self.B))

# 量化基座 + 第一个 Linear 加 LoRA
torch.manual_seed(123)
model = GPTModel(cfg)
quantize_model_int8(model)                    # 1. 量化基座
model.trf_blocks[0].att.W_query = LoRALayer(  # 2. 接 LoRA
    model.trf_blocks[0].att.W_query, r=8, alpha=16
)

idx = torch.tensor([tok.encode("Hello")])
model.eval()
with torch.no_grad():
    out = model(idx)
print(f"QLoRA 模型(量化基座+LoRA) 输出: {tuple(out.shape)} ✓")
print("\n💡 4bit 量化基座 + LoRA = QLoRA，单卡微调大模型的标准方案。")

---
> **小结**：量化用低位整数近似权重，省显存换可接受的误差。int8 几乎无损，int4 配 LoRA 成 QLoRA。
> 真实工程用 `bitsandbytes` / `GPTQ` / `AWQ` 等库做生产级量化。